# Лабораторная работа №1. KNN

Здесь я решаю две задачи: классифицирую ответ об употреблении каннабиса за последние 12 месяцев и прогнозирую число новых обращений 311 о граффити в районе Чикаго на следующую неделю. Для каждой задачи сравниваю обычный KNN из sklearn, его улучшенный вариант и такие же два варианта собственной реализации.

Источники: [Drug Consumption (UCI)](https://archive.ics.uci.edu/dataset/373/drug-consumption-quantified) и [Chicago Graffiti Removal Historical](https://data.cityofchicago.org/Service-Requests/311-Service-Requests-Graffiti-Removal-Historical/hec5-y4x5). Исходные файлы не хранятся в репозитории.


## 1. Загрузка данных

Данные скачиваются при первом запуске. В Colab после перезапуска среды загрузка повторится. Проверяю размер файлов, чтобы случайно не использовать 1000 строк из стандартного API-запроса Chicago.


In [ ]:
from pathlib import Path
from urllib.request import urlopen
from zipfile import ZipFile
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42
DATA_DIR = Path("datasets")
DATA_DIR.mkdir(exist_ok=True)
DRUG_PATH = DATA_DIR / "drug_consumption.data"
GRAFFITI_PATH = DATA_DIR / "Chicago_graffiti.csv"

def download(url, destination, min_bytes):
    if destination.exists() and destination.stat().st_size >= min_bytes:
        print(f"Уже есть: {destination} ({destination.stat().st_size / 1e6:.1f} МБ)")
        return
    temporary = destination.with_suffix(destination.suffix + ".part")
    with urlopen(url, timeout=120) as response, temporary.open("wb") as output:
        shutil.copyfileobj(response, output)
    if temporary.stat().st_size < min_bytes:
        temporary.unlink()
        raise RuntimeError(f"Файл {destination.name} слишком мал: возможно, скачана лишь часть данных")
    temporary.replace(destination)

if not DRUG_PATH.exists():
    archive = DATA_DIR / "drug_consumption.zip"
    download("https://archive.ics.uci.edu/static/public/373/drug+consumption+quantified.zip", archive, 100_000)
    with ZipFile(archive) as zf:
        member = next(name for name in zf.namelist() if name.endswith("drug_consumption.data"))
        with zf.open(member) as source, DRUG_PATH.open("wb") as target:
            shutil.copyfileobj(source, target)
download("https://data.cityofchicago.org/api/v3/views/hec5-y4x5/export.csv?accessType=DOWNLOAD", GRAFFITI_PATH, 100_000_000)
assert DRUG_PATH.stat().st_size > 100_000
assert GRAFFITI_PATH.stat().st_size > 100_000_000


## 2. Классификация: недавнее употребление каннабиса

`CL3`–`CL6` означают употребление за последний год или чаще. Остальные ответы считаю отрицательным классом. Это классификация ответа в анкете, а не прогноз будущего поведения. Использую только 12 признаков респондента: ID и ответы про другие вещества могли бы подменить смысл задачи.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, balanced_accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.preprocessing import StandardScaler

columns = ["id", "age", "gender", "education", "country", "ethnicity", "nscore", "escore", "oscore", "ascore", "cscore", "impulsive", "ss", "alcohol", "amphet", "amyl", "benzos", "caff", "cannabis", "choc", "coke", "crack", "ecstasy", "heroin", "ketamine", "legalh", "lsd", "meth", "mushrooms", "nicotine", "semer", "vsa"]
drug = pd.read_csv(DRUG_PATH, header=None, names=columns, skipinitialspace=True)
feature_columns = columns[1:13]
X_cls = drug[feature_columns].astype(float)
y_cls = drug["cannabis"].isin(["CL3", "CL4", "CL5", "CL6"]).astype(int)
assert len(drug) == 1885 and y_cls.sum() == 999
print("Размер:", X_cls.shape, "Доли классов:", y_cls.value_counts(normalize=True).round(3).to_dict())
display(X_cls.describe().round(2))


Тестовую часть откладываю сразу. Валидация нужна для выбора параметров; к тесту обращаюсь только после выбора модели. Стратификация сохраняет доли классов.


In [ ]:
X_dev_c, X_test_c, y_dev_c, y_test_c = train_test_split(X_cls, y_cls, test_size=0.2, stratify=y_cls, random_state=RANDOM_STATE)
X_train_c, X_val_c, y_train_c, y_val_c = train_test_split(X_dev_c, y_dev_c, test_size=0.25, stratify=y_dev_c, random_state=RANDOM_STATE)

def classification_scores(y_true, y_pred):
    return {"F1": f1_score(y_true, y_pred), "Balanced accuracy": balanced_accuracy_score(y_true, y_pred)}

majority = np.full(len(y_val_c), int(y_train_c.mode()[0]))
base_cls = KNeighborsClassifier(n_neighbors=5)
base_cls.fit(X_train_c, y_train_c)
print("Большинство:", classification_scores(y_val_c, majority))
print("KNN, k=5:", classification_scores(y_val_c, base_cls.predict(X_val_c)))


### Гипотезы для классификации

1. Масштабирование повысит качество: хотя признаки уже преобразованы авторами, их разброс различается.
2. Другое число соседей поможет сгладить шум.
3. Взвешивание по расстоянию даст больший вес более похожим респондентам.

Проверяю варианты на валидации. Основная метрика — F1 положительного класса; balanced accuracy показывает, насколько хорошо распознаются оба класса. Параметры преобразования обучаю только на обучающей части.


In [ ]:
rows = []
for scaled in [False, True]:
    scaler = StandardScaler() if scaled else None
    Xtr = scaler.fit_transform(X_train_c) if scaled else X_train_c.to_numpy()
    Xva = scaler.transform(X_val_c) if scaled else X_val_c.to_numpy()
    for k in [3, 5, 11, 21]:
        for weights in ["uniform", "distance"]:
            model = KNeighborsClassifier(n_neighbors=k, weights=weights)
            model.fit(Xtr, y_train_c)
            score = classification_scores(y_val_c, model.predict(Xva))
            rows.append({"scaled": scaled, "k": k, "weights": weights, **score})
cls_search = pd.DataFrame(rows).sort_values("F1", ascending=False)
display(cls_search.head(10).round(3))
best_c = cls_search.iloc[0]
print("Выбрано:", best_c[["scaled", "k", "weights"]].to_dict())


## 3. Регрессия: число обращений на следующую неделю

В исходной таблице одно обращение — одна строка. Сначала исключаю обращения, помеченные как повторы, некорректные даты и строки без района. Затем строю календарь для всех 77 Community Area, чтобы недели без обращений не исчезали. Дата строки — начало прогнозируемой недели; все признаки берутся только из прошлых недель.


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

graffiti = pd.read_csv(GRAFFITI_PATH, usecols=["Creation Date", "Status", "Service Request Number", "Community Area"], low_memory=False)
assert len(graffiti) > 1_000_000, "Похоже, API вернул неполный файл"
graffiti["date"] = pd.to_datetime(graffiti["Creation Date"], format="%m/%d/%Y", errors="coerce")
graffiti["area"] = pd.to_numeric(graffiti["Community Area"], errors="coerce")
graffiti = graffiti.drop_duplicates("Service Request Number")
graffiti = graffiti[graffiti["date"].between("2011-01-01", "2018-12-02") & graffiti["area"].between(1, 77) & ~graffiti["Status"].str.contains("Dup", na=False)].copy()
graffiti["area"] = graffiti["area"].astype(int)
graffiti["week"] = graffiti["date"].dt.to_period("W-SUN").dt.start_time
weekly = graffiti.groupby(["area", "week"]).size().rename("y")
calendar = pd.MultiIndex.from_product([range(1, 78), pd.date_range("2011-01-03", "2018-11-26", freq="W-MON")], names=["area", "week"])
weekly = weekly.reindex(calendar, fill_value=0).reset_index()
weekly = weekly.sort_values(["area", "week"]).reset_index(drop=True)
g = weekly.groupby("area")["y"]
weekly["lag_1"] = g.shift(1)
weekly["mean_3"] = g.transform(lambda s: s.shift(1).rolling(3).mean())
weekly["sum_4"] = g.transform(lambda s: s.shift(1).rolling(4).sum())
weekly["lag_52"] = g.shift(52)
weekly["week_sin"] = np.sin(2 * np.pi * weekly["week"].dt.isocalendar().week.astype(int) / 52.18)
weekly["week_cos"] = np.cos(2 * np.pi * weekly["week"].dt.isocalendar().week.astype(int) / 52.18)
weekly = weekly.dropna().reset_index(drop=True)
print("Обращений после фильтрации:", len(graffiti), "Строк модели:", len(weekly), "Районов:", weekly.area.nunique())
display(weekly.head())


Прогнозируем количество сообщений в службу 311, а не все реальные случаи граффити. Начало 2011 года теряется из-за признака «аналогичная неделя год назад». Разбиение строго по времени: обучение до 2017 года, подбор на 2017-м, тест на январе–ноябре 2018-го. Неполный декабрь не использую.


In [ ]:
train_r = weekly[weekly.week < "2017-01-01"].copy()
val_r = weekly[(weekly.week >= "2017-01-01") & (weekly.week < "2018-01-01")].copy()
test_r = weekly[(weekly.week >= "2018-01-01") & (weekly.week < "2018-12-01")].copy()
base_cols = ["area", "lag_1", "mean_3", "sum_4"]
improved_cols = ["lag_1", "mean_3", "sum_4", "lag_52", "week_sin", "week_cos"]
assert max(train_r.week) < min(val_r.week) < min(test_r.week)
print("train / validation / test:", len(train_r), len(val_r), len(test_r))
print("Наивный прогноз (прошлая неделя), MAE:", mean_absolute_error(val_r.y, val_r.lag_1))
base_reg = KNeighborsRegressor(n_neighbors=5)
base_reg.fit(train_r[base_cols], train_r.y)
print("KNN k=5, MAE:", mean_absolute_error(val_r.y, base_reg.predict(val_r[base_cols])))


На графике видно, как меняется недельное число обращений по всему городу. График помогает понять сезонность и общий тренд, но сам по себе не доказывает качество прогноза.


In [ ]:
city_weekly = weekly.groupby("week")["y"].sum()
ax = city_weekly.plot(figsize=(11, 3), title="Обращения о граффити по неделям")
ax.set(xlabel="Неделя", ylabel="Обращений")
plt.show()


### Гипотезы для регрессии

Числовой код района не является расстоянием между районами, поэтому в улучшенном варианте заменяю его one-hot признаками. Добавляю сезонность и число обращений в аналогичную неделю прошлого года. Числовые признаки масштабирую по обучающей части. Затем выбираю число соседей и тип весов по MAE на 2017 годе. RMSE покажет чувствительность к крупным ошибкам.


In [ ]:
def regression_features(frame, areas, scaler=None, fit=False):
    numeric = frame[improved_cols].to_numpy(dtype=float)
    if fit:
        scaler = StandardScaler().fit(numeric)
    numeric = scaler.transform(numeric)
    area_codes = pd.Categorical(frame.area, categories=areas).codes
    one_hot = np.eye(len(areas), dtype=float)[area_codes]
    return np.column_stack([numeric, one_hot]), scaler

all_areas = list(range(1, 78))
Xtr_r, scaler_r = regression_features(train_r, all_areas, fit=True)
Xva_r, _ = regression_features(val_r, all_areas, scaler_r)
rows = []
for k in [5, 15, 31]:
    for weights in ["uniform", "distance"]:
        model = KNeighborsRegressor(n_neighbors=k, weights=weights)
        model.fit(Xtr_r, train_r.y)
        pred = model.predict(Xva_r)
        rows.append({"k": k, "weights": weights, "MAE": mean_absolute_error(val_r.y, pred), "RMSE": np.sqrt(mean_squared_error(val_r.y, pred))})
reg_search = pd.DataFrame(rows).sort_values("MAE")
display(reg_search.round(2))
best_r = reg_search.iloc[0]
print("Выбрано:", best_r[["k", "weights"]].to_dict())


## 4. Своя реализация KNN

Модель запоминает обучающие объекты, пакетами вычисляет евклидовы расстояния до новых объектов и выбирает k ближайших. Для классификации использую голосование, для регрессии — среднее. Если вес зависит от расстояния, точные совпадения имеют приоритет. `sklearn.neighbors` внутри этой реализации не используется.


In [ ]:
class MyKNN:
    def __init__(self, k=5, weights="uniform", task="classification", batch_size=128):
        self.k, self.weights, self.task, self.batch_size = k, weights, task, batch_size

    def fit(self, X, y):
        self.X = np.asarray(X, dtype=float)
        self.y = np.asarray(y)
        if not 1 <= self.k <= len(self.X):
            raise ValueError("k должно быть между 1 и числом обучающих объектов")
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        predictions = []
        train_norm = np.sum(self.X ** 2, axis=1)
        for start in range(0, len(X), self.batch_size):
            batch = X[start:start + self.batch_size]
            squared = np.maximum(np.sum(batch ** 2, axis=1)[:, None] + train_norm[None, :] - 2 * batch @ self.X.T, 0)
            near = np.argpartition(squared, self.k - 1, axis=1)[:, :self.k]
            distances = np.sqrt(np.take_along_axis(squared, near, axis=1))
            targets = self.y[near]
            if self.weights == "uniform":
                w = np.ones_like(distances)
            elif self.weights == "distance":
                w = 1 / np.maximum(distances, 1e-12)
                zeros = distances < 1e-12
                w = np.where(zeros.any(axis=1, keepdims=True), zeros.astype(float), w)
            else:
                raise ValueError("Неизвестный тип весов")
            if self.task == "regression":
                out = (w * targets).sum(axis=1) / w.sum(axis=1)
            elif self.task == "classification":
                classes = np.unique(self.y)
                votes = np.stack([(w * (targets == c)).sum(axis=1) for c in classes], axis=1)
                out = classes[np.argmax(votes, axis=1)]
            else:
                raise ValueError("Неизвестная задача")
            predictions.extend(out)
        return np.asarray(predictions)


Сначала сверяю реализацию со sklearn на небольшом подмножестве. Это проверка корректности формул, а не оценка качества модели. Затем обучаю все четыре варианта на одинаковых итоговых наборах данных и оцениваю на отложенном тесте.


In [ ]:
check_x = X_train_c.iloc[:150].to_numpy()
check_y = y_train_c.iloc[:150].to_numpy()
check_test = X_val_c.iloc[:30].to_numpy()
assert np.array_equal(MyKNN(k=5).fit(check_x, check_y).predict(check_test), KNeighborsClassifier(n_neighbors=5).fit(check_x, check_y).predict(check_test))
check_targets = np.arange(150, dtype=float)
assert np.allclose(MyKNN(k=5, task="regression").fit(check_x, check_targets).predict(check_test), KNeighborsRegressor(n_neighbors=5).fit(check_x, check_targets).predict(check_test))
print("Проверка на подмножестве пройдена")


In [ ]:
Xdev_c = X_dev_c.to_numpy(dtype=float)
Xtest_c = X_test_c.to_numpy(dtype=float)
ydev_c = y_dev_c.to_numpy()
cls_results = []
for name, model in [
    ("sklearn, базовый", KNeighborsClassifier(n_neighbors=5)),
    ("свой, базовый", MyKNN(k=5)),
]:
    model.fit(Xdev_c, ydev_c)
    cls_results.append({"Вариант": name, **classification_scores(y_test_c, model.predict(Xtest_c))})

scaler_c = StandardScaler().fit(Xdev_c)
Xdev_scaled = scaler_c.transform(Xdev_c)
Xtest_scaled = scaler_c.transform(Xtest_c)
k_c, weights_c = int(best_c.k), best_c.weights
for name, model in [
    ("sklearn, улучшенный", KNeighborsClassifier(n_neighbors=k_c, weights=weights_c)),
    ("свой, улучшенный", MyKNN(k=k_c, weights=weights_c)),
]:
    Xfit, Xeval = (Xdev_scaled, Xtest_scaled) if bool(best_c.scaled) else (Xdev_c, Xtest_c)
    model.fit(Xfit, ydev_c)
    cls_results.append({"Вариант": name, **classification_scores(y_test_c, model.predict(Xeval))})
display(pd.DataFrame(cls_results).round(3))


In [ ]:
dev_r = pd.concat([train_r, val_r]).sort_values(["area", "week"])
reg_results = []
for name, model in [
    ("sklearn, базовый", KNeighborsRegressor(n_neighbors=5)),
    ("свой, базовый", MyKNN(k=5, task="regression")),
]:
    model.fit(dev_r[base_cols].to_numpy(dtype=float), dev_r.y.to_numpy())
    pred = model.predict(test_r[base_cols].to_numpy(dtype=float))
    reg_results.append({"Вариант": name, "MAE": mean_absolute_error(test_r.y, pred), "RMSE": np.sqrt(mean_squared_error(test_r.y, pred))})

Xdev_r, final_scaler_r = regression_features(dev_r, all_areas, fit=True)
Xtest_r, _ = regression_features(test_r, all_areas, final_scaler_r)
k_r, weights_r = int(best_r.k), best_r.weights
for name, model in [
    ("sklearn, улучшенный", KNeighborsRegressor(n_neighbors=k_r, weights=weights_r)),
    ("свой, улучшенный", MyKNN(k=k_r, weights=weights_r, task="regression")),
]:
    model.fit(Xdev_r, dev_r.y.to_numpy())
    pred = model.predict(Xtest_r)
    reg_results.append({"Вариант": name, "MAE": mean_absolute_error(test_r.y, pred), "RMSE": np.sqrt(mean_squared_error(test_r.y, pred))})
print("Наивный прогноз, тест MAE:", mean_absolute_error(test_r.y, test_r.lag_1))
display(pd.DataFrame(reg_results).round(2))


## 5. Выводы

При локальном запуске на скачанных файлах базовый KNN для классификации получил F1 = 0.823 и balanced accuracy = 0.811 на тесте. На валидации лучший вариант (21 сосед, веса по расстоянию, без дополнительного масштабирования) поднял F1 с 0.822 до 0.837. На тесте его F1 оказался ниже базового: 0.802. Значит, улучшение на валидации здесь не обобщилось; подбирать параметры заново по тесту нельзя. Собственная реализация дала те же значения, что sklearn.

Для регрессии базовый KNN получил MAE = 11.83 обращения, улучшенный — 11.44. Наивный прогноз по прошлой неделе дал MAE = 13.10. Значит, KNN полезнее простого переноса прошлой недели, а сезонные признаки, кодирование района и подбор соседей помогли. Собственная улучшенная модель почти точно повторила sklearn; у базовой MAE немного отличается (11.85 против 11.83), вероятно из-за выбора соседей при одинаковых расстояниях. RMSE около 21–22 показывает, что встречаются недели с большими ошибками.

Данные 311 отражают число обращений, а не все случаи граффити. При повторном скачивании портала числа могут немного измениться, поэтому выводы нужно сверить с таблицами текущего запуска.
